In [1]:
import numpy as np
from numba import njit

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import scipy.sparse as sp

from sklearn.linear_model import RidgeCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error

import optuna
import optuna.visualization as vis
from optuna.importance import PedAnovaImportanceEvaluator

import cProfile
import pstats

In [2]:
steps = 20000

tau_steps = 1

transient_steps_henon = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_henon + transient_steps_reservoir + tau_steps
total_steps_after_henon = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

# Init

In [3]:
henon_dataset = np.zeros((total_steps, 2))

rng = np.random.default_rng(42)
henon_dataset[0] = rng.random(2)

a = 1.4
b = 0.3

In [4]:
@njit
def henon_numba(steps, a=1.4, b=0.3, x0=0.0, y0=0.0):
    X = np.zeros(steps)
    Y = np.zeros(steps)
    X[0] = x0
    Y[0] = y0

    for i in range(1, steps):
        X[i] = 1 - a * X[i - 1] ** 2 + Y[i - 1]
        Y[i] = b * X[i - 1]

    return X, Y

In [5]:
henon_data_x, henon_data_y = henon_numba(total_steps)

henon_dataset = np.column_stack((henon_data_x, henon_data_y))
henon_dataset = henon_dataset[transient_steps_henon:]

In [6]:
henon_scaler = StandardScaler()
henon_scaled = henon_scaler.fit_transform(henon_dataset)

In [7]:
def henon_plot(data_list, names=None):
    fig = go.Figure()
    colors = ["white", "magenta"]

    for i, data in enumerate(data_list):
        fig.add_trace(
            go.Scatter(
                x=data[:, 0],
                y=data[:, 1],
                mode="markers",
                name=names[i] if names else f"Dataset {i+1}",
                marker=dict(color=colors[i % len(colors)], size=1),
            )
        )

    fig.update_layout(
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white"),
        xaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
        yaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
    )

    return fig

In [8]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(go.Scatter(
            x=actual, y=predicted, mode="markers",
            name="Data", marker=dict(color="rgba(50, 50, 200, 0.5)", size=5)
        ), row=1, col=col)

        min_val, max_val = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
        fig.add_trace(go.Scatter(
            x=[min_val, max_val], y=[min_val, max_val], mode="lines", 
            name="Ideal", line=dict(color="firebrick", dash="dash")
        ), row=1, col=col)

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [9]:
def param_importance(study):
    top_5_percent_evaluator = PedAnovaImportanceEvaluator(target_quantile=0.05)

    fig1 = vis.plot_param_importances(
        study,
        target=lambda t: t.values[0] * -1.0,
        target_name="R2 Score",
        evaluator=top_5_percent_evaluator,
    )
    fig2 = vis.plot_param_importances(
        study,
        target=lambda t: t.values[1],
        target_name="MSE",
        evaluator=top_5_percent_evaluator,
    )

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Parameter Importance (R²)", "Parameter Importance (MSE)"),
        horizontal_spacing=0.2,
    )

    fig.add_trace(fig1.data[0], row=1, col=1)
    fig.add_trace(fig2.data[0], row=1, col=2)

    fig.update_layout(height=500, width=1000, showlegend=False, paper_bgcolor="white")

    fig.update_xaxes(title_text="Importance", row=1, col=1)
    fig.update_xaxes(title_text="Importance", row=1, col=2)
    fig.update_yaxes(categoryorder="total ascending", row=1, col=1)
    fig.update_yaxes(categoryorder="total ascending", row=1, col=2)

    return fig

# Henon Open

## Custom Vals

In [10]:
in_size = 2
out_size = 2
res_size = 300
sparsity = 0.1
spec_rad = 1.0
alpha = 0.7
input_scaling = 2

In [11]:
rng = np.random.default_rng(42)

bias = rng.uniform(-input_scaling, input_scaling, res_size)

W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))

W_res = rng.uniform(-1.0, 1.0, (res_size, res_size))
mask = rng.random((res_size, res_size)) > sparsity
W_res[mask] = 0.0

v0 = rng.normal(size=W_res.shape[0])
eigenvalues = np.linalg.eigvals(W_res)
largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)

W_res = W_res * (spec_rad / largest_eigenvalue)

In [12]:
@njit
def compute_states(steps, henon_inputs, res_size, W_in, W_res, bias, alpha):
    X_data = np.zeros((steps, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = henon_inputs @ W_in.T

    for i in range(1, steps):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data

X = compute_states(
    total_steps_after_henon, henon_scaled, res_size, W_in, W_res, bias, alpha
)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [13]:
X_data = X_scaled[transient_steps_reservoir:-tau_steps]
Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]

X_train, X_test = (
    X_data[:-test_steps],
    X_data[-test_steps:],
)
Y_train, Y_test_unscaled = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

model = RidgeCV()
model.fit(X_train, Y_train)
Y_pred_scaled = model.predict(X_test)

Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_unscaled)

In [14]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)

r_2, mse

(0.9998840960444129, 0.005683688411988077)

In [15]:
henon_plot([henon_dataset[-Y_pred.shape[0] :], Y_pred], ["Test", "Pred"]).show()
r_2_plots_grid(Y_test, Y_pred, ["X", "Y"]).show()

## Noise

If someone bumps a pendulum the RC and learn the modifies inputs or ignore it.

In [ ]:
in_size = 2
out_size = 2
res_size = 300
sparsity = 0.1
spec_rad = 1.0
alpha = 0.7
input_scaling = 2
noise_val = .1

In [ ]:
rng = np.random.default_rng(42)

bias = rng.uniform(-input_scaling, input_scaling, res_size)

W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))

W_res = rng.uniform(-1.0, 1.0, (res_size, res_size))
mask = rng.random((res_size, res_size)) > sparsity
W_res[mask] = 0.0

v0 = rng.normal(size=W_res.shape[0])
eigenvalues = np.linalg.eigvals(W_res)
largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)

W_res = W_res * (spec_rad / largest_eigenvalue)

In [ ]:
@njit
def compute_states(steps, henon_inputs, res_size, W_in, W_res, bias, alpha, noise):
    X_data = np.zeros((steps, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = (henon_inputs + noise) @ W_in.T

    for i in range(1, steps):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data

rng = np.random.default_rng(42)
noise = rng.normal(
    0, noise_val, size=(len(henon_scaled), out_size)
).astype(np.float32)

X = compute_states(
    total_steps_after_henon, henon_scaled, res_size, W_in, W_res, bias, alpha, noise
)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_data = X_scaled[transient_steps_reservoir:-tau_steps]
Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]

X_train, X_test = (
    X_data[:-test_steps],
    X_data[-test_steps:],
)
Y_train, Y_test_unscaled = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

model = RidgeCV()
model.fit(X_train, Y_train)
Y_pred_scaled = model.predict(X_test)

Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_unscaled)

In [188]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)

r_2, mse

(0.9689780115167795, 0.09085540718555976)

In [189]:
henon_plot([henon_dataset[-Y_pred.shape[0] :], Y_pred], ["Test", "Pred"]).show()
r_2_plots_grid(Y_test, Y_pred, ["X", "Y"]).show()

## Static Noise

We kinda end up adding energy to the motion. So we kinda end up actually changing the position of the pendulum.

In [190]:
in_size = 2
out_size = 2
res_size = 300
sparsity = 0.1
spec_rad = 1.0
alpha = 0.7
input_scaling = 2
static_noise = 0.1

In [ ]:
rng = np.random.default_rng(42)

bias = rng.uniform(-input_scaling, input_scaling, res_size)

W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))

W_res = rng.uniform(-1.0, 1.0, (res_size, res_size))
mask = rng.random((res_size, res_size)) > sparsity
W_res[mask] = 0.0

v0 = rng.normal(size=W_res.shape[0])
eigenvalues = np.linalg.eigvals(W_res)
largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)

W_res = W_res * (spec_rad / largest_eigenvalue)

In [ ]:
@njit
def compute_states(steps, henon_inputs, res_size, W_in, W_res, bias, alpha, noise):
    X_data = np.zeros((steps, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = henon_inputs @ W_in.T

    for i in range(1, steps):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias + noise[i - 1]
        )
        X_data[i] = state
    return X_data


rng = np.random.default_rng(42)
noise = rng.normal(0, static_noise, size=(test_steps, res_size)).astype(np.float32)

X = compute_states(
    total_steps_after_henon, henon_scaled, res_size, W_in, W_res, bias, alpha, noise
)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_data = X_scaled[transient_steps_reservoir:-tau_steps]
Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]

X_train, X_test = (
    X_data[:-test_steps],
    X_data[-test_steps:],
)
Y_train, Y_test_unscaled = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

model = RidgeCV()
model.fit(X_train, Y_train)
Y_pred_scaled = model.predict(X_test)

Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_unscaled)

In [195]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)

r_2, mse

(0.9988449451738247, 0.018028782814417745)

In [196]:
henon_plot([henon_dataset[-Y_pred.shape[0] :], Y_pred], ["Test", "Pred"]).show()
r_2_plots_grid(Y_test, Y_pred, ["X", "Y"]).show()

# Henon Open Bayesian

## Custom Vals

In [10]:
@njit
def compute_states(steps, henon_inputs, res_size, W_in, W_res, bias, alpha):
    X_data = np.zeros((steps, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = henon_inputs @ W_in.T

    for i in range(1, steps):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data

In [35]:
def henon_closed(
    in_size,
    out_size,
    res_size,
    sparsity,
    spec_rad,
    alpha,
    input_scaling,
    ridge_alpha,
    tau_steps,
):
    rng = np.random.default_rng(42)
    bias = rng.uniform(-input_scaling, input_scaling, res_size)
    W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))
    W_res_sparse = sp.random(
            res_size,
            res_size,
            density=sparsity,
            format="csr",
            random_state=rng,
            data_rvs=lambda l: rng.uniform(-1.0, 1.0, l),
        )
    eigenvalues, _ = sp.linalg.eigs(W_res_sparse, k=1, which="LM", ncv=20)
    largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
    W_res_sparse = W_res_sparse * (spec_rad / largest_eigenvalue)
    W_res_dense = W_res_sparse.toarray()

    X = compute_states(
        total_steps_after_henon, henon_scaled, res_size, W_in, W_res_dense, bias, alpha
    )
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_data = X_scaled[transient_steps_reservoir:-tau_steps]
    Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]
    X_train, X_test = (
        X_data[:-test_steps],
        X_data[-test_steps:],
    )
    Y_train, Y_test_unscaled = (
        Y_data[:-test_steps],
        Y_data[-test_steps:],
    )

    model = Ridge(alpha=ridge_alpha, solver="cholesky")
    model.fit(X_train, Y_train)
    Y_pred_scaled = model.predict(X_test)
    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
    Y_test = henon_scaler.inverse_transform(Y_test_unscaled)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return r_2, mse, Y_test, Y_pred

In [36]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    r_2, mse, *_ = henon_closed(
        in_size=2,
        out_size=2,
        res_size=trial.suggest_int("res_size", 10, 2000),
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 0.1, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-15, 10.0, log=True),
        tau_steps=1,
    )
    return r_2, mse


study = optuna.create_study(directions=["maximize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=20, n_jobs=-1)

[Optuna] Processing Trial #19...

In [37]:
best_trials = study.best_trials

print("Pareto Front trials:")
for trial in best_trials:
    print(f"Trial {trial.number}: R2={trial.values[0]}, MSE={trial.values[1]}")
    print(f"Params: {trial.params}")

Pareto Front trials:
Trial 11: R2=0.9999999720721724, MSE=8.71677588030372e-05
Params: {'res_size': 785, 'sparsity': 0.2855624603848331, 'spec_rad': 0.3448237070399601, 'alpha': 0.47218900215662646, 'input_scaling': 1.9132889611967923, 'ridge_alpha': 9.332072611049095e-12}


In [39]:
params = study.best_trials[0].params
r_2, mse, Y_test, Y_pred = henon_closed(
    in_size=2,
    out_size=2,
    res_size=params["res_size"],
    sparsity=params["sparsity"],
    spec_rad=params["spec_rad"],
    alpha=params["alpha"],
    input_scaling=params["input_scaling"],
    ridge_alpha=params["ridge_alpha"],
    tau_steps=1,
)
r_2, mse

(0.9999999720721724, 8.71677588030372e-05)

In [40]:
params = study.best_trials[0].params

profiler = cProfile.Profile()
profiler.enable()

henon_closed(
    in_size=2,
    out_size=2,
    res_size=params["res_size"],
    sparsity=params["sparsity"],
    spec_rad=params["spec_rad"],
    alpha=params["alpha"],
    input_scaling=params["input_scaling"],
    ridge_alpha=params["ridge_alpha"],
    tau_steps=1,
)

profiler.disable()

stats = pstats.Stats(profiler).sort_stats("tottime")
stats.print_stats(15);

         63331 function calls (63323 primitive calls) in 2.783 seconds

   Ordered by: internal time
   List reduced from 528 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    2.067    2.067    2.782    2.782 /var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_67565/3207060466.py:1(henon_closed)
     1047    0.309    0.000    0.309    0.000 {built-in method scipy.sparse._sparsetools.csr_matvec}
        2    0.064    0.032    0.064    0.032 /Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:166(safe_sparse_dot)
       37    0.061    0.002    0.061    0.002 {method 'reduce' of 'numpy.ufunc' objects}
     1049    0.050    0.000    0.050    0.000 {built-in method scipy.sparse.linalg._eigen.arpack._arpacklib.dnaupd_wrap}
        1    0.040    0.040    0.092    0.092 /Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/sklearn/utils/e

In [41]:
henon_plot([henon_dataset[-Y_pred.shape[0] :], Y_pred], ["Test", "Pred"]).show()
r_2_plots_grid(Y_test, Y_pred, ["X", "Y"]).show()
vis.plot_pareto_front(study, target_names=["R2 Score", "MSE"]).show()
param_importance(study).show()

/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_67565/3029333622.py:2: ExperimentalWarning: PedAnovaImportanceEvaluator is experimental (supported from v3.6.0). The interface can change in the future.
  top_5_percent_evaluator = PedAnovaImportanceEvaluator(target_quantile=0.05)
/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_67565/3029333622.py:4: UserWarning: PedAnovaImportanceEvaluator computes the importances of params to achieve low `target` values. If this is not what you want, please modify target, e.g., by multiplying the output by -1.
  fig1 = vis.plot_param_importances(
/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_67565/3029333622.py:10: UserWarning: PedAnovaImportanceEvaluator computes the importances of params to achieve low `target` values. If this is not what you want, please modify target, e.g., by multiplying the output by -1.
  fig2 = vis.plot_param_importances(


## Bias

In [60]:
@njit
def compute_states(steps, henon_inputs, res_size, W_in, W_res, bias, alpha):
    X_data = np.zeros((steps, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = henon_inputs @ W_in.T

    for i in range(1, steps):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data

In [63]:
def henon_closed(
    in_size,
    out_size,
    res_size,
    sparsity,
    spec_rad,
    alpha,
    input_scaling,
    bias_scaling,
    ridge_alpha,
    tau_steps,
):
    rng = np.random.default_rng(42)
    bias = rng.uniform(-bias_scaling, bias_scaling, res_size)
    W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))
    W_res_sparse = sp.random(
            res_size,
            res_size,
            density=sparsity,
            format="csr",
            random_state=rng,
            data_rvs=lambda l: rng.uniform(-1.0, 1.0, l),
        )
    eigenvalues, _ = sp.linalg.eigs(W_res_sparse, k=1, which="LM", ncv=20)
    largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
    W_res_sparse = W_res_sparse * (spec_rad / largest_eigenvalue)
    W_res_dense = W_res_sparse.toarray()

    X = compute_states(
        total_steps_after_henon, henon_scaled, res_size, W_in, W_res_dense, bias, alpha
    )
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_data = X_scaled[transient_steps_reservoir:-tau_steps]
    Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]
    X_train, X_test = (
        X_data[:-test_steps],
        X_data[-test_steps:],
    )
    Y_train, Y_test_unscaled = (
        Y_data[:-test_steps],
        Y_data[-test_steps:],
    )

    model = Ridge(alpha=ridge_alpha, solver="cholesky")
    model.fit(X_train, Y_train)
    Y_pred_scaled = model.predict(X_test)
    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
    Y_test = henon_scaler.inverse_transform(Y_test_unscaled)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return r_2, mse, Y_test, Y_pred

In [64]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    r_2, mse, *_ = henon_closed(
        in_size=2,
        out_size=2,
        res_size=trial.suggest_int("res_size", 10, 2000),
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 0.1, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 0.1, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 10.0, log=True),
        tau_steps=1,
    )
    return r_2, mse


study = optuna.create_study(directions=["maximize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=20, n_jobs=-1)

[Optuna] Processing Trial #16...

/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 4.0592753783857147e-17.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


[Optuna] Processing Trial #19...

In [65]:
best_trials = study.best_trials

print("Pareto Front trials:")
for trial in best_trials:
    print(f"Trial {trial.number}: R2={trial.values[0]}, MSE={trial.values[1]}")
    print(f"Params: {trial.params}")

Pareto Front trials:
Trial 19: R2=0.9999999999994059, MSE=3.9444380622361703e-07
Params: {'res_size': 1225, 'sparsity': 0.34048710619941963, 'spec_rad': 0.2333525902348053, 'alpha': 0.7924213343132849, 'input_scaling': 0.3664898577674777, 'bias_scaling': 1.2565907923793431, 'ridge_alpha': 4.879142205156299e-12}


In [68]:
params = study.best_trials[0].params
r_2, mse, Y_test, Y_pred = henon_closed(
    in_size=2,
    out_size=2,
    res_size=params["res_size"],
    sparsity=params["sparsity"],
    spec_rad=params["spec_rad"],
    alpha=params["alpha"],
    input_scaling=params["input_scaling"],
    bias_scaling=params["bias_scaling"],
    ridge_alpha=params["ridge_alpha"],
    tau_steps=1,
)
r_2, mse

(0.9999999999994059, 3.9444380622361703e-07)

In [69]:
henon_plot([henon_dataset[-Y_pred.shape[0] :], Y_pred], ["Test", "Pred"]).show()
r_2_plots_grid(Y_test, Y_pred, ["X", "Y"]).show()
vis.plot_pareto_front(study, target_names=["R2 Score", "MSE"]).show()
param_importance(study).show()

/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_67565/3029333622.py:2: ExperimentalWarning: PedAnovaImportanceEvaluator is experimental (supported from v3.6.0). The interface can change in the future.
  top_5_percent_evaluator = PedAnovaImportanceEvaluator(target_quantile=0.05)
/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_67565/3029333622.py:4: UserWarning: PedAnovaImportanceEvaluator computes the importances of params to achieve low `target` values. If this is not what you want, please modify target, e.g., by multiplying the output by -1.
  fig1 = vis.plot_param_importances(
/var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_67565/3029333622.py:10: UserWarning: PedAnovaImportanceEvaluator computes the importances of params to achieve low `target` values. If this is not what you want, please modify target, e.g., by multiplying the output by -1.
  fig2 = vis.plot_param_importances(


# Henon Closed

## Custom Vals

In [368]:
in_size = 2
out_size = 2
res_size = 1000
sparsity = 0.3
spec_rad = 0.9
alpha = 0.3
input_scaling = 0.3
bias_scaling = 1.2
ridge_alpha = 4e-5

In [369]:
rng = np.random.default_rng(42)

bias = rng.uniform(-bias_scaling, bias_scaling, res_size)

W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))

W_res_sparse = sp.random(
    res_size,
    res_size,
    density=sparsity,
    format="csr",
    random_state=rng,
    data_rvs=lambda n: rng.uniform(-1.0, 1.0, n),
)
eigenvalues, _ = sp.linalg.eigs(W_res_sparse, k=1, which="LM", ncv=20)
largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)

W_res = W_res_sparse * (spec_rad / largest_eigenvalue)
W_res_dense = W_res.toarray()

In [370]:
@njit
def compute_states(steps, henon_inputs, res_size, W_in, W_res, bias, alpha):
    X_data = np.zeros((steps, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = henon_inputs @ W_in.T

    for i in range(1, steps):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data


X = compute_states(
    steps + transient_steps_reservoir - test_steps,
    henon_scaled[: -test_steps - tau_steps],
    res_size,
    W_in,
    W_res_dense,
    bias,
    alpha,
)

In [371]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X[transient_steps_reservoir:])
Y_train = henon_scaled[transient_steps_reservoir + tau_steps : -test_steps]

In [372]:
model = Ridge(alpha=ridge_alpha)
model.fit(X_train, Y_train)
W_out = model.coef_
W_bias = model.intercept_

In [373]:
@njit
def compute_closed_states(
    steps, Y_pred_scaled, X_pred, W_in, W_res, bias, alpha, W_out, W_bias, res_size
):
    for i in range(2, steps):
        u = Y_pred_scaled[i - 2]
        prev_state = X_pred[i - 1, :res_size]
        new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
        X_pred[i, :res_size] = (1.0 - alpha) * prev_state + alpha * new_state
        Y_pred_scaled[i] = W_out @ X_pred[i] + W_bias
    return Y_pred_scaled

In [374]:
X_pred = np.zeros((test_steps, res_size))
henon_scaled_test = henon_scaled[-test_steps - tau_steps : -tau_steps]
Y_pred_scaled = np.zeros((test_steps, out_size))
Y_test = henon_dataset[-test_steps:]

u = Y_train[-2]
prev_state = X_train[-1]
new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
X_pred[0] = (1 - alpha) * prev_state + alpha * new_state
Y_pred_scaled[0] = model.predict(X_pred[0].reshape(1, -1))

u = Y_train[-1]
prev_state = X_pred[0]
new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
X_pred[1] = (1 - alpha) * prev_state + alpha * new_state
Y_pred_scaled[1] = model.predict(X_pred[1].reshape(1, -1))

compute_closed_states(
    test_steps,
    Y_pred_scaled,
    X_pred,
    W_in,
    W_res_dense,
    bias,
    alpha,
    W_out,
    W_bias,
    res_size,
)

Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)

In [375]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)

r_2, mse

(-123.56902786985532, 5.823885915875121)

In [376]:
henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()
r_2_plots_grid(Y_test, Y_pred, ["X", "Y"]).show()

## Noise

In [377]:
in_size = 2
out_size = 2
res_size = 1000
sparsity = 0.3
spec_rad = 0.9
alpha = 0.3
input_scaling = 0.3
bias_scaling = 1.2
noise_val = 0.1
ridge_alpha = 4e-5

In [378]:
rng = np.random.default_rng(42)

bias = rng.uniform(-bias_scaling, bias_scaling, res_size)

W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))

W_res_sparse = sp.random(
    res_size,
    res_size,
    density=sparsity,
    format="csr",
    random_state=rng,
    data_rvs=lambda n: rng.uniform(-1.0, 1.0, n),
)
eigenvalues, _ = sp.linalg.eigs(W_res_sparse, k=1, which="LM", ncv=20)
largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)

W_res = W_res_sparse * (spec_rad / largest_eigenvalue)
W_res_dense = W_res.toarray()

In [379]:
@njit
def compute_states(steps, henon_inputs, res_size, W_in, W_res, bias, alpha, noise):
    X_data = np.zeros((steps, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = (henon_inputs + noise) @ W_in.T

    for i in range(1, steps):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data

rng = np.random.default_rng(42)
noise = rng.normal(
    0, noise_val, size=(len(henon_scaled) - test_steps - tau_steps, out_size)
)

X = compute_states(
    steps + transient_steps_reservoir - test_steps,
    henon_scaled[: -test_steps - tau_steps],
    res_size,
    W_in,
    W_res_dense,
    bias,
    alpha,
    noise,
)

In [380]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X[transient_steps_reservoir:])
Y_train = henon_scaled[transient_steps_reservoir + tau_steps : -test_steps]

In [381]:
model = Ridge(alpha=ridge_alpha)
model.fit(X_train, Y_train)
W_out = model.coef_
W_bias = model.intercept_

In [382]:
@njit
def compute_closed_states(
    steps, Y_pred_scaled, X_pred, W_in, W_res, bias, alpha, W_out, W_bias, res_size
):
    for i in range(2, steps):
        u = Y_pred_scaled[i - 2]
        prev_state = X_pred[i - 1, :res_size]
        new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
        X_pred[i, :res_size] = (1.0 - alpha) * prev_state + alpha * new_state
        Y_pred_scaled[i] = W_out @ X_pred[i] + W_bias
    return Y_pred_scaled

In [383]:
X_pred = np.zeros((test_steps, res_size))
henon_scaled_test = henon_scaled[-test_steps - tau_steps : -tau_steps]
Y_pred_scaled = np.zeros((test_steps, out_size))
Y_test = henon_dataset[-test_steps:]

u = Y_train[-2]
prev_state = X_train[-1]
new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
X_pred[0] = (1 - alpha) * prev_state + alpha * new_state
Y_pred_scaled[0] = model.predict(X_pred[0].reshape(1, -1))

u = Y_train[-1]
prev_state = X_pred[0]
new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
X_pred[1] = (1 - alpha) * prev_state + alpha * new_state
Y_pred_scaled[1] = model.predict(X_pred[1].reshape(1, -1))

compute_closed_states(
    test_steps, Y_pred_scaled, X_pred, W_in, W_res_dense, bias, alpha, W_out, W_bias, res_size
)

Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)

In [387]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)

r_2, mse

(-48527.11699285272, 115.89347140360132)

In [388]:
henon_plot([Y_test, Y_pred], ["Test", "Pred"]).show()
r_2_plots_grid(Y_test, Y_pred, ["X", "Y"]).show()

# Bayesian Open

In [33]:
@njit
def compute_states(steps, henon_inputs, res_size, W_in, W_res, bias, alpha, noise):
    X_data = np.zeros((steps, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = (henon_inputs + noise) @ W_in.T

    for i in range(1, steps):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data

In [34]:
@njit
def compute_closed_states(
    steps, Y_pred_scaled, X_pred, W_in, W_res, bias, alpha, W_out, W_bias, res_size, scaler_mean, scaler_std
):
    for i in range(2, steps):
        u = Y_pred_scaled[i - 2]
        prev_state = X_pred[i - 1, :res_size]
        new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
        X_pred[i, :res_size] = (1.0 - alpha) * prev_state + alpha * new_state
        X_scaled = (X_pred[i, :res_size] - scaler_mean) / scaler_std
        Y_pred_scaled[i] = W_out @ X_scaled + W_bias
    return Y_pred_scaled

In [41]:
def henon_closed(
    in_size,
    out_size,
    res_size,
    sparsity,
    spec_rad,
    alpha,
    input_scaling,
    bias_scaling,
    ridge_alpha,
    noise_val,
    tau_steps,
):
    rng = np.random.default_rng(42)
    bias = rng.uniform(-bias_scaling, bias_scaling, res_size)
    W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))
    W_res_sparse = sp.random(
        res_size,
        res_size,
        density=sparsity,
        format="csr",
        random_state=rng,
        data_rvs=lambda l: rng.uniform(-1.0, 1.0, l),
    )
    try:
        eigenvalues, _ = sp.linalg.eigs(W_res_sparse, k=1, which="LM", ncv=20)
    except Exception as e:
        raise optuna.exceptions.TrialPruned()
    largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
    W_res = W_res_sparse * (spec_rad / largest_eigenvalue)
    W_res_dense = W_res.toarray()

    rng = np.random.default_rng(42)
    noise = rng.normal(
        0, noise_val, size=(len(henon_scaled) - test_steps - tau_steps, out_size)
    )
    X = compute_states(
        steps + transient_steps_reservoir - test_steps,
        henon_scaled[: -test_steps - tau_steps],
        res_size,
        W_in,
        W_res_dense,
        bias,
        alpha,
        noise,
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[transient_steps_reservoir:])
    Y_train = henon_scaled[transient_steps_reservoir + tau_steps : -test_steps]
    scaler_mean = scaler.mean_
    scaler_std = scaler.scale_

    model = Ridge(alpha=ridge_alpha)
    model.fit(X_train, Y_train)
    W_out = model.coef_
    W_bias = model.intercept_

    X_pred = np.zeros((test_steps, res_size))
    Y_pred_scaled = np.zeros((test_steps, out_size))
    Y_test = henon_dataset[-test_steps:]

    u = Y_train[-2]
    prev_state = X_train[-1]
    new_state = np.tanh(W_in @ u + W_res_dense @ prev_state + bias)
    X_pred[0] = (1 - alpha) * prev_state + alpha * new_state
    X_pred_scaled = scaler.transform(X_pred[0].reshape(1, -1))
    Y_pred_scaled[0] = model.predict(X_pred_scaled)

    u = Y_train[-1]
    prev_state = X_pred[0]
    new_state = np.tanh(W_in @ u + W_res_dense @ prev_state + bias)
    X_pred[1] = (1 - alpha) * prev_state + alpha * new_state
    X_pred_scaled = scaler.transform(X_pred[1].reshape(1, -1))
    Y_pred_scaled[1] = model.predict(X_pred_scaled)

    compute_closed_states(
        test_steps, Y_pred_scaled, X_pred, W_in, W_res_dense, bias, alpha, W_out, W_bias, res_size, scaler_mean, scaler_std
    )

    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return r_2, mse, Y_test, Y_pred

In [42]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    _, mse, *_ = henon_closed(
        in_size=2,
        out_size=2,
        res_size=trial.suggest_int("res_size", 10, 2000),
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 0.1, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 0.1, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 10.0, log=True),
        noise_val=trial.suggest_float("noise_val", 1e-6, .1, log=True),
        tau_steps=1,
    )
    return mse


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=20, n_jobs=-1)

[Optuna] Processing Trial #19...

In [43]:
print(f"MSE={study.best_value}")
print(f"Params: {study.best_params}")

MSE=1.7778335607806022
Params: {'res_size': 1029, 'sparsity': 0.2574901127695319, 'spec_rad': 1.5877579455808035, 'alpha': 0.1893363832905451, 'input_scaling': 0.3682324174081161, 'bias_scaling': 0.12649307942512292, 'ridge_alpha': 1.57744743054136e-13, 'noise_val': 0.026004699766120642}


In [44]:
params = study.best_trials[0].params
r_2, mse, Y_test, Y_pred = henon_closed(
    in_size=2,
    out_size=2,
    res_size=params["res_size"],
    sparsity=params["sparsity"],
    spec_rad=params["spec_rad"],
    alpha=params["alpha"],
    input_scaling=params["input_scaling"],
    bias_scaling=params["bias_scaling"],
    ridge_alpha=params["ridge_alpha"],
    noise_val=params["noise_val"],
    tau_steps=1,
)
r_2, mse

(-17.322288750663592, 1.7778335607806022)

In [45]:
henon_plot([henon_dataset[-Y_pred.shape[0] :], Y_pred], ["Test", "Pred"]).show()
r_2_plots_grid(Y_test, Y_pred, ["X", "Y"]).show()